In [ ]:
# CREATE CSV

from pathlib import Path

import pandas as pd
from astropy.io import fits
from data_types import Header_Content, S4ACS_Config
from utils import create_csv

config = S4ACS_Config.from_config_file("acs_config.cfg")
nights_folder = config.image_path / ".."

nights = [
    night.stem
    for night in nights_folder.iterdir()
    if "2026" in night.stem and "test" not in night.stem and "temp" not in night.stem
]
for night in nights[68:]:
    csv_file = Path.cwd() / "_csv" / f"{night}.csv"
    create_csv(csv_file)
    config = S4ACS_Config.from_config_file("acs_config.cfg")
    images_folder = config.image_path / ".." / night
    images_list = [
        name.resolve()
        for name in images_folder.iterdir()
        if name.suffix == ".fits" and "test" not in name.name
    ]

    for image_file in images_list:
        hdr = fits.getheader(image_file)
        hdr_cnt = Header_Content.from_header(hdr)
        df = pd.DataFrame([hdr_cnt.to_csv_format])
        df.to_csv(csv_file, mode="a", index=False, header=False)

In [9]:
# CREATE REPORT

import pandas as pd
from pathlib import Path

headers = ["Date", "N. images", "Bias", "Flat", "Phot", "Polar"]
report = {k: [] for k in headers}

csv_folder = Path.cwd() / "_csv"
obs_nights = csv_folder.iterdir()
for night in obs_nights:
    report["Date"].append(night.stem)
    df = pd.read_csv(csv_folder / night)
    n_images = df.shape[0]
    report["N. images"].append(n_images)
    bias = df[df["OBSTYPE"] == "ZERO"].shape[0]
    report["Bias"].append(bias)
    flat = df[df["OBSTYPE"] == "FLAT"].shape[0]
    report["Flat"].append(flat)
    inst_mode = df[df["INSTMODE"] == "PHOT"].shape[0]
    report["Phot"].append(inst_mode)
    report["Polar"].append(n_images - inst_mode)

report = pd.DataFrame.from_dict(report)
report = report.sort_values("Polar", ascending=False)
report_file = Path.cwd() / "report.csv"
report.to_csv(report_file, index=False)
print(report.head(n=10))

        Date  N. images  Bias  Flat  Phot  Polar
51  20260731      10881   324   240     0  10881
30  20260702       9607   300   500   348   9259
43  20260719       8026   300   495   315   7711
56  20260805       6911   300   480     0   6911
38  20260714       5256   300   604   399   4857
39  20260715       4567   300   510   575   3992
54  20260803       4136   300   495   315   3821
55  20260804       4053   300   480   300   3753
47  20260727       4079   446     0   947   3132
66  20260816       3770   300   255   697   3073
